### ML Project

In [ ]:
# import necessary libraries
import pandas as pd
import numpy as np
import re

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score


'.' expected
import pandas as pd
             ^
';' expected
import pandas as pd
                ^


In [ ]:
# Download necessary NLTK resources
nltk.download("punkt")
nltk.download("stopwords")


In [ ]:
# Load dataset
df = pd.read_csv(
    "Arabic-Tweets_Dataset.csv",
    encoding="utf-8",
    errors="ignore"
)


In [ ]:
# Load Arabic stopwords
arabic_stopwords = set(stopwords.words("arabic"))

In [ ]:
# Initialize spell checker
spell = SpellChecker(language="ar")

In [ ]:
# Emoji handling
def count_emojis(text):
    return emoji.emoji_count(text)

def handle_emojis(text):
    return emoji.replace_emoji(text, replace=" EMOJI ")

In [ ]:
# Function to normalize Arabic text
def normalize_arabic(text):
    text = strip_tashkeel(text)      # remove tashkeel
    text = strip_tatweel(text)       # remove ~
    text = normalize_hamza(text)     # normalize hamza
    text = normalize_ligature(text)  # لا forms
    return text

In [ ]:
# Tokenization
def tokenize(text):
    return word_tokenize(text)

In [ ]:
# Negation handling
NEGATIONS = {"لا", "لم", "لن", "ما", "مش", "مو", "ليس", "بدون", "غير", "دون", "سوى", "عدا", "الا" }

def handle_negation(tokens):
    result = []
    negate = False
    for word in tokens:
        if word in NEGATIONS:
            negate = True
            result.append(word)
        elif negate:
            result.append("NEG_" + word)
            negate = False
        else:
            result.append(word)
    return result

In [ ]:
# Precompute spell corrections for all unique tokens ---
print("Precomputing spell corrections...")
all_tokens = set()
for text in df["text"]:
    tokens = tokenize(normalize_arabic(handle_emojis(str(text).lower())))
    all_tokens.update(tokens)

corrected_dict = {w: spell.correction(w) or w for w in all_tokens}

def correct_tokens(tokens):
    return [corrected_dict.get(t, t) if not t.startswith("NEG_") else t for t in tokens]


In [ ]:
# Main preprocessing function
def preprocess_text(text):
    text = str(text).lower()
    emoji_count = count_emojis(text)
    text = handle_emojis(text)
    text = normalize_arabic(text)
    
    tokens = tokenize(text)
    tokens = handle_negation(tokens)
    tokens = [t for t in tokens if t not in arabic_stopwords]
    tokens = correct_tokens(tokens)
    
    clean_text = " ".join(tokens)
    return clean_text, emoji_count

In [ ]:
# Apply preprocessing
print("Applying preprocessing to dataset...")
df[["clean_text", "emoji_count"]] = df["text"].apply(lambda x: pd.Series(preprocess_text(x)))

In [ ]:
# Feature extraction
def extract_features(tokens, emoji_count):
    return {
        "tweet_length": len(tokens),
        "has_negation": int(any(t.startswith("NEG_") for t in tokens)),
        "emoji_count": emoji_count
    }

features_df = pd.DataFrame(
    df.apply(
        lambda row: extract_features(
            row["clean_text"].split(),
            row["emoji_count"]
        ),
        axis=1
    ).tolist()
)

In [ ]:
# Combine features with original dataframe
df_final = pd.concat([df, features_df], axis=1)

In [ ]:
# Save preprocessed data
df_final.to_csv("arabic_sentiment_preprocessed_person_b.csv", index=False)

In [ ]:
# Completion message
print("Preprocessing complete. File saved as:")
print("arabic_sentiment_preprocessed_person_b.csv")